# YouTube → SRT (ruso) con faster-whisper

Notebook autocontenido. No depende del resto del repo Sta-RU.

**Qué hace:** descarga audio de una lista de links de YouTube, los transcribe a SRT en ruso con `large-v2`, y te entrega todo en un ZIP. Suena un ruidito cuando termina.

**Son 2 celdas:** la primera prepara todo y te pide los links; la segunda hace el trabajo y te baja el ZIP.

**Antes de correr:** `Runtime → Change runtime type → T4 GPU` (o cualquier GPU). Sin GPU también corre, pero mucho más lento.


## 1) Setup + pegá tus links

Corré esta celda. Instala todo y chequea que YouTube no te esté bloqueando:

- Si ves **`ALL OK`** en verde, aparece abajo el cuadro para pegar tus links (uno por línea). Pegálos y pasá al paso 2.
- Si ves el cartel **rojo** de bloqueo, no aparece el cuadro: cambiá de IP (o reconectá el runtime) y volvé a correr esta celda.

Admite videos sueltos y playlists. Las líneas que arrancan con `#` se ignoran.


In [ ]:
!pip install -q faster-whisper yt-dlp deep-translator
!apt-get -qq install -y ffmpeg > /dev/null

import torch, os, shutil, zipfile, time, re
import yt_dlp
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "int8_float16" if DEVICE == "cuda" else "int8"
print(f"Device: {DEVICE}  |  compute_type: {COMPUTE_TYPE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

YTDLP_CLIENTS = ["default", "tv_simply", "mweb", "android_vr", "tv", "web_safari"]

def _youtube_reachable():
    canary = "https://www.youtube.com/watch?v=jNQXAC9IVRw"
    for client in YTDLP_CLIENTS:
        opts = {"quiet": True, "no_warnings": True, "skip_download": True, "simulate": True}
        if client != "default":
            opts["extractor_args"] = {"youtube": {"player_client": [client]}}
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                if ydl.extract_info(canary, download=False):
                    return True
        except Exception:
            continue
    return False

print("\nChequeando acceso a YouTube...")
if _youtube_reachable():
    print("\033[1;92mALL OK\033[0m  — pegá tus links abajo:\n")
    url_box = widgets.Textarea(
        value="",
        placeholder="Pegá acá tus links de YouTube, uno por línea:\nhttps://www.youtube.com/watch?v=...\nhttps://www.youtube.com/watch?v=...",
        layout=widgets.Layout(width="95%", height="180px"),
        continuous_update=True,
    )
    display(url_box)
    print("\n↑ Pegá tus links (uno por línea) y corré la celda del paso 2.")
else:
    url_box = None
    print("\033[1;91mYOUTUBE IS BLOCKING YOU, CHANGE YOUR IP\033[0m")
    print("No avanzo: cambiá de IP (o reconectá el runtime) y volvé a correr esta celda.")


## 2) Procesar todo (descargar → transcribir → ZIP)

Una sola celda hace el resto:

1. **Descarga** el audio de cada link (deduplica links repetidos por id).
2. **Traduce** el título al inglés y arma el nombre `Fecha - Título en inglés-RU.srt`. Si dos videos distintos chocan en el nombre, agrega el id de YouTube solo a los que chocan.
3. **Transcribe** a SRT en ruso con `large-v2` (mismos parámetros que tu llamada de PowerShell: `--temperature 0 --beam_size 5 --best_of 1 --task transcribe`). Si el SRT ya existe, lo salta.
4. **Empaqueta** los SRT de esta corrida en un ZIP, suena un ruidito y dispara la descarga.

El modelo se carga una sola vez: si re-corrés la celda (p. ej. para sumar links), reusa el que ya está en memoria.


In [ ]:
from collections import Counter
from deep_translator import GoogleTranslator
from faster_whisper import WhisperModel
from IPython.display import Audio, display
import numpy as np
from google.colab import files

# ---- Leer los links del cuadro del paso 1 ----
assert globals().get("url_box") is not None, \
    "Primero corré el paso 1. Si te apareció el cartel rojo de bloqueo, cambiá de IP y volvé a correrlo."
URLS = [u.strip() for u in url_box.value.strip().splitlines() if u.strip() and not u.strip().startswith("#")]
assert URLS, "El cuadro de links está vacío. Subí al paso 1, pegá los links y corré esta celda otra vez."

AUDIO_DIR = Path("/content/audios")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
MEDIA_EXTS = {".m4a", ".webm", ".opus", ".mp3", ".mp4", ".wav", ".ogg", ".aac", ".mkv"}

_FORBIDDEN = re.compile(r'[\\/:*?"<>|\n\r\t]')
def sanitize(name):
    return _FORBIDDEN.sub("_", name).strip().rstrip(". ") or "untitled"

def fmt_date(raw):
    if raw and len(raw) == 8 and raw.isdigit():
        return f"{raw[:4]}-{raw[4:6]}-{raw[6:]}"
    return raw or ""

def fmt_ts(t):
    h = int(t // 3600)
    m = int((t % 3600) // 60)
    s = int(t % 60)
    ms = int(round((t - int(t)) * 1000))
    if ms == 1000:
        ms = 0
        s += 1
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def ydl_run(url, download):
    """extract_info ciclando player clients para esquivar el bot-check de IP cloud."""
    last = "unknown"
    for client in YTDLP_CLIENTS:
        opts = {
            "outtmpl": str(AUDIO_DIR / "%(id)s.%(ext)s"),
            "format": "bestaudio[ext=m4a]/bestaudio/best",
            "ignoreerrors": True, "quiet": True, "no_warnings": True,
        }
        if client != "default":
            opts["extractor_args"] = {"youtube": {"player_client": [client]}}
        try:
            with yt_dlp.YoutubeDL(opts) as ydl:
                info = ydl.extract_info(url, download=download)
            if info:
                return info
            last = "no info (posible bot-check)"
        except Exception as ex:
            last = str(ex)
    raise RuntimeError(f"yt-dlp falló para {url}: {last}")

translator = GoogleTranslator(source="auto", target="en")
def translate_en(title):
    if not title:
        return ""
    try:
        return (translator.translate(title) or title).strip()
    except Exception as ex:
        print(f"   [WARN] no pude traducir el título ({ex}); uso el original")
        return title

# ---- 1) Descargar + dedup por video id ----
print(f"Descargando {len(URLS)} link(s)...")
entries = {}  # vid -> {audio, title, date}
for url in URLS:
    info = ydl_run(url, download=True)
    for ent in (info.get("entries") or [info]):
        if not ent:
            continue
        vid = ent.get("id")
        if not vid or vid in entries:
            continue
        matches = sorted(p for p in AUDIO_DIR.glob(f"{vid}.*") if p.suffix.lower() in MEDIA_EXTS)
        if not matches:
            print(f"   [WARN] no encontré el audio descargado para id={vid}")
            continue
        entries[vid] = {
            "audio": matches[0],
            "title": ent.get("title", "") or "",
            "date": fmt_date(ent.get("upload_date", "")),
        }

# ---- 2) Traducir títulos + nombres únicos ----
for vid, e in entries.items():
    e["stem"] = " - ".join(p for p in [e["date"], translate_en(e["title"])] if p) or vid
stem_counts = Counter(e["stem"] for e in entries.values())
jobs = []  # cada job: {'audio': Path, 'srt': Path}
for vid, e in entries.items():
    stem = e["stem"] if stem_counts[e["stem"]] == 1 else f"{e['stem']} [{vid}]"
    jobs.append({"audio": e["audio"], "srt": AUDIO_DIR / f"{sanitize(stem)}-RU.srt"})
assert jobs, "No se descargó ningún audio. ¿YouTube te bloqueó? Volvé al paso 1."

print(f"\n{len(jobs)} video(s) a transcribir:")
for j in jobs:
    print("  -", j["srt"].name)

# ---- 3) Transcribir (el modelo se carga una sola vez y se reusa en re-runs) ----
if "model" not in globals():
    print("\nCargando modelo large-v2 (la primera vez tarda)...")
    model = WhisperModel("large-v2", device=DEVICE, compute_type=COMPUTE_TYPE)
    print("Modelo listo.")

total = len(jobs)
t_global = time.time()
for i, job in enumerate(jobs, 1):
    audio, srt = job["audio"], job["srt"]
    if srt.exists():
        print(f"\n[{i}/{total}] SALTADO (ya existe): {srt.name}")
        continue
    print(f"\n[{i}/{total}] Procesando: {srt.name}")
    t0 = time.time()
    segments, info = model.transcribe(
        str(audio),
        language="ru",
        task="transcribe",
        temperature=0,
        beam_size=5,
        best_of=1,
        vad_filter=True,
        condition_on_previous_text=True,
    )
    print(f"   duración audio: {info.duration:.1f}s")
    n = 0
    with open(srt, "w", encoding="utf-8") as f:
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            n += 1
            f.write(f"{n}\n{fmt_ts(seg.start)} --> {fmt_ts(seg.end)}\n{text}\n\n")
    print(f"[{i}/{total}] Listo ({n} líneas, {time.time()-t0:.1f}s)")
print(f"\nTranscripción terminada en {(time.time()-t_global)/60:.1f} min.")

# ---- 4) Empaquetar ZIP (solo los SRT de esta corrida) + ruidito + descarga ----
srts = [j["srt"] for j in jobs if j["srt"].exists()]
missing = [j["srt"].name for j in jobs if not j["srt"].exists()]
ZIP_PATH = "/content/subtitulos_ru.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for s in srts:
        zf.write(s, arcname=s.name)
print(f"\nZIP listo: {ZIP_PATH}  ({len(srts)} SRT, {os.path.getsize(ZIP_PATH)/1024:.1f} KB)")
if missing:
    print("[WARN] faltan estos SRT (¿falló la transcripción?):")
    for m in missing:
        print("  -", m)

# Ruidito: fanfarria ascendente sol-do-mi-sol-do
sr = 22050
out = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    env = np.exp(-3*t)
    out = np.concatenate([out, (0.3*env*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out, rate=sr, autoplay=True))

files.download(ZIP_PATH)
